# DQN Stable-Baselines3 sur highway-v0 - Run "defaults highway-env"

Ce notebook entraine SB3 DQN avec les hyperparametres **recommandes officiellement par la doc highway-env** pour SB3, sans alignement sur le DQN custom. Il complete `sb3_training.ipynb` (run aligne) et permet de departager:

1. l'effet des **hyperparametres** (gamma, train_freq, exploration schedule, etc.)
2. l'effet des **details d'implementation** (optimiseur, batching) deja teste dans le run aligne

Source des hyperparametres: https://highway-env.farama.org/quickstart/

In [ ]:
import os, json, datetime, glob
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import highway_env
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from shared_core_config import SHARED_CORE_ENV_ID, SHARED_CORE_CONFIG

## 01. Configuration

Hyperparametres **strictement** conformes a la doc highway-env. Aucun override.

| Parametre | Valeur | Source |
|---|---|---|
| `learning_rate` | 5e-4 | doc highway-env |
| `buffer_size` | 15 000 | doc highway-env |
| `learning_starts` | 200 | doc highway-env |
| `batch_size` | 32 | doc highway-env |
| `gamma` | **0.8** | doc highway-env |
| `train_freq` | **1** | doc highway-env |
| `gradient_steps` | **1** | doc highway-env |
| `target_update_interval` | 50 | doc highway-env |
| `net_arch` | [256, 256] | doc highway-env |
| `exploration_fraction` | 0.1 | SB3 default (non specifie par highway) |
| `exploration_final_eps` | 0.05 | SB3 default (non specifie par highway) |
| `total_timesteps` | 40 000 | budget compute equivalent au run aligne (~35-40k steps) |

**Note sur gamma=0.8**: l'ablation sur le DQN custom a montre que gamma=0.99 donne de bien meilleurs resultats que gamma=0.8 (20.20 vs 18.23 de reward, 2.67% vs 74% de crash rate). On garde quand meme gamma=0.8 pour rester fidele a la recommandation officielle. C'est exactement le genre de question que ce run permet de soulever: **la valeur recommandee par la doc est-elle reellement optimale ?**

**Note sur total_timesteps=40 000**: la doc highway-env utilise `int(2e4)` (20k steps) dans son exemple minimal, mais le run aligne tourne sur ~35-40k steps. Pour une comparaison "compute-matched" equitable, on donne a SB3 defaults le meme budget de steps.

In [ ]:
# Seeds et offsets identiques aux autres runs (custom + SB3 aligne)
SEEDS = [0, 1, 2]
EVAL_EPISODES = 50
EVAL_SEED_OFFSET = 1_000
FAILURE_SEED_OFFSET = 9_000
FAILURE_EPISODES = 100

# Hyperparams highway-env defaults
LR = 5e-4
BUFFER_SIZE = 15_000
LEARNING_STARTS = 200
BATCH_SIZE = 32
GAMMA = 0.8
TRAIN_FREQ = 1
GRADIENT_STEPS = 1
TARGET_UPDATE = 50
NET_ARCH = [256, 256]

# SB3 defaults pour l'exploration (non specifies par highway-env doc)
EXPLORATION_FRACTION = 0.1
EXPLORATION_FINAL_EPS = 0.05
EXPLORATION_INITIAL_EPS = 1.0

# Budget compute aligne sur le run aligne (~35-40k steps observes)
TOTAL_TIMESTEPS = 40_000

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
SAVE_DIR = os.path.join("checkpoints_sb_defaults", f"run_{RUN_ID}")
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

## 02. Environnement

In [ ]:
def make_env(seed=0):
    env = gym.make(SHARED_CORE_ENV_ID, render_mode="rgb_array")
    env.unwrapped.configure(SHARED_CORE_CONFIG)
    env.reset(seed=seed)
    return env

## 03. Entrainement multi-seed

Logger minimal des rewards par episode. **Pas de callback d'override d'epsilon**, pas de callback de stop: on laisse SB3 tourner en mode natif sur `total_timesteps`.

In [ ]:
class EpisodeRewardLogger(BaseCallback):
    """Logge les rewards et longueurs par episode, sans modifier le comportement de SB3."""

    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_epsilons = []
        self._ep_reward = 0.0
        self._ep_length = 0

    def _on_step(self):
        self._ep_reward += self.locals["rewards"][0]
        self._ep_length += 1

        if self.locals["dones"][0]:
            self.episode_rewards.append(self._ep_reward)
            self.episode_lengths.append(self._ep_length)
            self.episode_epsilons.append(float(self.model.exploration_rate))
            self._ep_reward = 0.0
            self._ep_length = 0
        return True


trained_models = {}
training_data = {}

for seed in SEEDS:
    print(f"\n{'='*50}")
    print(f"  Seed {seed}")
    print(f"{'='*50}")

    env = make_env(seed=seed)

    model = DQN(
        "MlpPolicy",
        env,
        learning_rate=LR,
        buffer_size=BUFFER_SIZE,
        learning_starts=LEARNING_STARTS,
        batch_size=BATCH_SIZE,
        gamma=GAMMA,
        train_freq=TRAIN_FREQ,
        gradient_steps=GRADIENT_STEPS,
        target_update_interval=TARGET_UPDATE,
        exploration_initial_eps=EXPLORATION_INITIAL_EPS,
        exploration_fraction=EXPLORATION_FRACTION,
        exploration_final_eps=EXPLORATION_FINAL_EPS,
        policy_kwargs=dict(net_arch=NET_ARCH),
        device=DEVICE,
        seed=seed,
        verbose=0,
    )

    logger = EpisodeRewardLogger()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=logger, progress_bar=True)

    model.save(os.path.join(SAVE_DIR, f"dqn_seed{seed}"))
    trained_models[seed] = model
    training_data[seed] = {
        "rewards": logger.episode_rewards,
        "lengths": logger.episode_lengths,
        "epsilons": logger.episode_epsilons,
    }
    env.close()

    print(f"  {len(logger.episode_rewards)} episodes, {TOTAL_TIMESTEPS} steps")

## 04. Courbes d'entrainement

In [ ]:
window = 100
fig, axes = plt.subplots(1, len(SEEDS), figsize=(7 * len(SEEDS), 5), sharey=True)
if len(SEEDS) == 1:
    axes = [axes]

for ax, seed in zip(axes, SEEDS):
    r = np.array(training_data[seed]["rewards"])
    ax.plot(r, alpha=0.3, label="par episode")
    if len(r) >= window:
        smooth = np.convolve(r, np.ones(window) / window, mode="valid")
        ax.plot(np.arange(window - 1, len(r)), smooth, label=f"MA-{window}")
    ax.set_title(f"SB3 DQN defaults - seed {seed}")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.legend()

fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "sb3_defaults_training_curves.png"), dpi=150)
plt.show()

## 04b. Courbe agregee multi-seed

In [ ]:
def plot_multi_seed(all_rewards, label="", window=100, ax=None, color=None):
    smoothed = []
    for rewards in all_rewards:
        t = np.array(rewards, dtype=np.float32)
        if len(t) >= window:
            s = np.convolve(t, np.ones(window) / window, mode="valid")
        else:
            s = t
        smoothed.append(s)

    min_len = min(len(s) for s in smoothed)
    arr = np.array([s[:min_len] for s in smoothed])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    x = np.arange(min_len) + window

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))

    kw = dict(color=color) if color else {}
    ax.plot(x, mean, label=label, **kw)
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, **kw)
    return ax


fig, ax = plt.subplots(figsize=(10, 5))
all_r = [training_data[s]["rewards"] for s in SEEDS]
plot_multi_seed(all_r, label="SB3 DQN defaults", ax=ax, color="seagreen")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward (MA-100)")
ax.set_title("SB3 DQN defaults - Training reward (mean +/- std across seeds)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "sb3_defaults_training_aggregated.png"), dpi=150)
plt.show()

## 05. Evaluation (50 episodes par seed)

Memes seeds d'evaluation que le run aligne et le DQN custom (`offset=1000`).

In [ ]:
def evaluate_sb3(model, n_episodes=50, seed_offset=1000):
    env = make_env(seed=seed_offset)
    rewards, lengths, crashes = [], [], []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed_offset + ep)
        done, total_r, steps = False, 0.0, 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_r += r
            steps += 1
            done = terminated or truncated
        rewards.append(total_r)
        lengths.append(steps)
        crashes.append(bool(info.get("crashed", False)))

    env.close()
    return {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "mean_length": float(np.mean(lengths)),
        "std_length": float(np.std(lengths)),
        "crash_rate": float(np.mean(crashes)),
        "raw_rewards": [float(x) for x in rewards],
    }


eval_results = {}
for seed in SEEDS:
    res = evaluate_sb3(trained_models[seed], n_episodes=EVAL_EPISODES,
                       seed_offset=EVAL_SEED_OFFSET)
    eval_results[seed] = res
    print(f"seed={seed}  reward={res['mean_reward']:.2f} +/- {res['std_reward']:.2f}  "
          f"crash_rate={res['crash_rate']:.0%}  length={res['mean_length']:.1f}")

## 06. Sauvegarde des resultats

Format JSON identique au run aligne pour permettre une comparaison croisee facile.

In [ ]:
summary = {
    "algorithm": "SB3 DQN defaults (highway-env)",
    "config": {
        "net_arch": NET_ARCH,
        "batch_size": BATCH_SIZE,
        "gamma": GAMMA,
        "lr": LR,
        "learning_starts": LEARNING_STARTS,
        "buffer_size": BUFFER_SIZE,
        "target_update": TARGET_UPDATE,
        "train_freq": TRAIN_FREQ,
        "gradient_steps": GRADIENT_STEPS,
        "exploration_initial_eps": EXPLORATION_INITIAL_EPS,
        "exploration_final_eps": EXPLORATION_FINAL_EPS,
        "exploration_fraction": EXPLORATION_FRACTION,
        "total_timesteps": TOTAL_TIMESTEPS,
        "eval_episodes": EVAL_EPISODES,
    },
    "seeds": SEEDS,
    "per_seed": [],
}

for seed in SEEDS:
    td = training_data[seed]
    ev = eval_results[seed]
    summary["per_seed"].append({
        "seed": seed,
        "training_episodes": len(td["rewards"]),
        "training_steps": int(np.sum(td["lengths"])),
        "eval": ev,
    })

all_means = [eval_results[s]["mean_reward"] for s in SEEDS]
all_crashes = [eval_results[s]["crash_rate"] for s in SEEDS]
summary["aggregate"] = {
    "mean_reward": float(np.mean(all_means)),
    "std_reward": float(np.std(all_means)),
    "crash_rate": float(np.mean(all_crashes)),
}

summary["all_rewards"] = [[float(r) for r in training_data[s]["rewards"]] for s in SEEDS]

metrics_path = os.path.join(SAVE_DIR, "sb3_defaults_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Metrics saved -> {metrics_path}")

agg = summary["aggregate"]
print(f"\nCross-seed: reward={agg['mean_reward']:.2f} +/- {agg['std_reward']:.2f}")
print(f"Cross-seed crash rate: {agg['crash_rate']:.0%}")

## 07. Graphique d'evaluation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

means = [eval_results[s]["mean_reward"] for s in SEEDS]
stds = [eval_results[s]["std_reward"] for s in SEEDS]
crash_rates = [eval_results[s]["crash_rate"] for s in SEEDS]

x = np.arange(len(SEEDS))
ax1.bar(x, means, yerr=stds, capsize=5, color="seagreen")
ax1.set_xticks(x)
ax1.set_xticklabels([f"seed {s}" for s in SEEDS])
ax1.set_ylabel("Reward moyen")
ax1.set_title("SB3 DQN defaults - Reward (50 eval episodes)")

ax2.bar(x, crash_rates, color="indianred")
ax2.set_xticks(x)
ax2.set_xticklabels([f"seed {s}" for s in SEEDS])
ax2.set_ylabel("Taux de crash")
ax2.set_ylim(0, 1)
ax2.set_title("SB3 DQN defaults - Crashes")

fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "sb3_defaults_eval_results.png"), dpi=150)
plt.show()

## 08. Video du meilleur seed

In [ ]:
best_seed = SEEDS[int(np.argmax(means))]
best_model = trained_models[best_seed]

video_dir = os.path.join(SAVE_DIR, f"rollout_seed{best_seed}")
os.makedirs(video_dir, exist_ok=True)

env = gym.make(SHARED_CORE_ENV_ID, render_mode="rgb_array")
env.unwrapped.configure(SHARED_CORE_CONFIG)
env = gym.wrappers.RecordVideo(env, video_dir, episode_trigger=lambda e: e < 3)

for ep in range(3):
    obs, _ = env.reset(seed=best_seed + ep)
    done = False
    total_r = 0.0
    while not done:
        action, _ = best_model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(int(action))
        total_r += r
        done = terminated or truncated
    print(f"Episode {ep}: reward={total_r:.3f}")

env.close()
print(f"Videos saved -> {video_dir}")

## 09. Analyse des echecs

In [ ]:
def analyse_failures(model, n_episodes=100, seed_offset=9000):
    env = make_env(seed=seed_offset)
    crash_episodes = []

    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed_offset + i)
        done, total_r, steps = False, 0.0, 0
        actions = []

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            action = int(action)
            obs, r, terminated, truncated, info = env.step(action)
            total_r += r
            steps += 1
            actions.append(action)
            done = terminated or truncated

        if bool(info.get("crashed", False)):
            crash_episodes.append({
                "episode": i,
                "seed": seed_offset + i,
                "total_reward": float(total_r),
                "length": steps,
            })

    env.close()
    return crash_episodes


crashes = analyse_failures(best_model, n_episodes=FAILURE_EPISODES,
                           seed_offset=FAILURE_SEED_OFFSET)

crash_path = os.path.join(SAVE_DIR, "failure_summaries.json")
with open(crash_path, "w") as f:
    json.dump(crashes, f, indent=2)

print(f"Crashes: {len(crashes)}/{FAILURE_EPISODES} episodes ({len(crashes)/FAILURE_EPISODES:.0%})")
if crashes:
    crash_rewards = [c["total_reward"] for c in crashes]
    crash_lengths = [c["length"] for c in crashes]
    print(f"  Reward moyen au crash: {np.mean(crash_rewards):.2f} +/- {np.std(crash_rewards):.2f}")
    print(f"  Longueur moyenne: {np.mean(crash_lengths):.1f} +/- {np.std(crash_lengths):.1f}")
    print(f"  Crash le plus precoce: step {min(crash_lengths)}")
    print(f"  Crash le plus tardif: step {max(crash_lengths)}")
print(f"Saved -> {crash_path}")

## 10. Comparaison croisee : Custom vs SB3 aligne vs SB3 defaults

Charge automatiquement les metriques des deux autres runs et produit un tableau et des graphiques a 3 colonnes.

In [ ]:
# Charger DQN custom (le plus recent run avec gamma=0.99)
custom_data = None
for p in sorted(glob.glob("checkpoints/run_*/gamma_0.99_metrics.json"), reverse=True):
    with open(p) as f:
        custom_data = json.load(f)
    print(f"Custom DQN charge: {p}")
    break
if custom_data is None:
    print("WARN: aucun custom metrics trouve")

# Charger SB3 aligne (le plus recent run dans checkpoints_sb)
sb3_aligned_data = None
for p in sorted(glob.glob("checkpoints_sb/run_*/sb3_metrics.json"), reverse=True):
    with open(p) as f:
        sb3_aligned_data = json.load(f)
    print(f"SB3 aligne charge: {p}")
    break
if sb3_aligned_data is None:
    print("WARN: aucun SB3 aligne metrics trouve")

In [ ]:
if custom_data is not None and sb3_aligned_data is not None:
    print("=" * 80)
    print("  TABLEAU COMPARATIF FINAL : Custom vs SB3 aligne vs SB3 defaults")
    print("=" * 80)
    print(f"{'Metric':<22} {'Custom DQN':>16} {'SB3 aligne':>16} {'SB3 defaults':>16}")
    print("-" * 80)

    cust_a = custom_data["aggregate"]
    sb3a_a = sb3_aligned_data["aggregate"]
    sb3d_a = summary["aggregate"]

    print(f"{'Mean reward':<22} {cust_a['mean_reward']:>16.2f} {sb3a_a['mean_reward']:>16.2f} {sb3d_a['mean_reward']:>16.2f}")
    print(f"{'Std reward (cross)':<22} {cust_a['std_reward']:>16.2f} {sb3a_a['std_reward']:>16.2f} {sb3d_a['std_reward']:>16.2f}")
    print(f"{'Crash rate':<22} {cust_a['crash_rate']:>15.1%} {sb3a_a['crash_rate']:>15.1%} {sb3d_a['crash_rate']:>15.1%}")
    print("-" * 80)

    cust_per_seed = {entry["seed"]: entry["eval"] for entry in custom_data["per_seed"]}
    sb3a_per_seed = {entry["seed"]: entry["eval"] for entry in sb3_aligned_data["per_seed"]}
    sb3d_per_seed = {entry["seed"]: entry["eval"] for entry in summary["per_seed"]}

    for seed in SEEDS:
        cs = cust_per_seed.get(seed, {})
        sa = sb3a_per_seed.get(seed, {})
        sd = sb3d_per_seed.get(seed, {})
        print(f"  Seed {seed}:")
        print(f"    {'reward':<18} {cs.get('mean_reward', 0):>16.2f} {sa.get('mean_reward', 0):>16.2f} {sd.get('mean_reward', 0):>16.2f}")
        print(f"    {'crash_rate':<18} {cs.get('crash_rate', 0):>15.1%} {sa.get('crash_rate', 0):>15.1%} {sd.get('crash_rate', 0):>15.1%}")
    print("=" * 80)

In [ ]:
if custom_data is not None and sb3_aligned_data is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    labels = ["Custom\nDQN", "SB3\naligne", "SB3\ndefaults"]
    colors = ["darkorange", "steelblue", "seagreen"]

    # 1) Reward agregee
    means_agg = [
        custom_data["aggregate"]["mean_reward"],
        sb3_aligned_data["aggregate"]["mean_reward"],
        summary["aggregate"]["mean_reward"],
    ]
    stds_agg = [
        custom_data["aggregate"]["std_reward"],
        sb3_aligned_data["aggregate"]["std_reward"],
        summary["aggregate"]["std_reward"],
    ]
    axes[0].bar(labels, means_agg, yerr=stds_agg, capsize=5, color=colors)
    axes[0].set_ylabel("Mean Reward (cross-seed)")
    axes[0].set_title("Reward agregee")

    # 2) Crash rate
    crash_agg = [
        custom_data["aggregate"]["crash_rate"],
        sb3_aligned_data["aggregate"]["crash_rate"],
        summary["aggregate"]["crash_rate"],
    ]
    axes[1].bar(labels, crash_agg, color=colors)
    axes[1].set_ylabel("Crash rate")
    axes[1].set_ylim(0, max(crash_agg) * 1.3 + 0.05)
    axes[1].set_title("Crash rate agrege")

    # 3) Reward par seed (groupes)
    x = np.arange(len(SEEDS))
    w = 0.27
    cust_means = [cust_per_seed.get(s, {}).get("mean_reward", 0) for s in SEEDS]
    sb3a_means = [sb3a_per_seed.get(s, {}).get("mean_reward", 0) for s in SEEDS]
    sb3d_means = [sb3d_per_seed.get(s, {}).get("mean_reward", 0) for s in SEEDS]

    axes[2].bar(x - w, cust_means, w, label="Custom", color="darkorange")
    axes[2].bar(x, sb3a_means, w, label="SB3 aligne", color="steelblue")
    axes[2].bar(x + w, sb3d_means, w, label="SB3 defaults", color="seagreen")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([f"seed {s}" for s in SEEDS])
    axes[2].set_ylabel("Mean Reward")
    axes[2].set_title("Reward par seed")
    axes[2].legend()

    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, "comparison_three_way.png"), dpi=150)
    plt.show()
    print(f"Saved -> {os.path.join(SAVE_DIR, 'comparison_three_way.png')}")

In [ ]:
# Courbes d'entrainement superposees
if custom_data is not None and sb3_aligned_data is not None and "all_rewards" in custom_data:
    fig, ax = plt.subplots(figsize=(12, 5))

    plot_multi_seed(custom_data["all_rewards"], label="Custom DQN", ax=ax, color="darkorange")
    if "all_rewards" in sb3_aligned_data:
        plot_multi_seed(sb3_aligned_data["all_rewards"], label="SB3 aligne", ax=ax, color="steelblue")
    plot_multi_seed([training_data[s]["rewards"] for s in SEEDS],
                    label="SB3 defaults", ax=ax, color="seagreen")

    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward (MA-100)")
    ax.set_title("Training curves: Custom vs SB3 aligne vs SB3 defaults")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, "comparison_training_curves_three_way.png"), dpi=150)
    plt.show()
    print(f"Saved -> {os.path.join(SAVE_DIR, 'comparison_training_curves_three_way.png')}")